In [1]:
!pip install -q chromadb pymupdf groq

In [2]:
import chromadb
import fitz
import groq
print("✅ All libraries working!")

✅ All libraries working!


In [3]:
from google.colab import files
import os

uploaded = files.upload()
os.makedirs("/content/data", exist_ok=True)
for fname in uploaded.keys():
    os.rename(fname, f"/content/data/{fname}")
print("✅ Uploaded:", os.listdir("/content/data"))

Saving Aishani Resume.pdf to Aishani Resume.pdf
✅ Uploaded: ['Aishani Resume.pdf']


In [4]:
import fitz
import os

def load_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def chunk_text(text, chunk_size=300, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []
for f in os.listdir("/content/data"):
    if f.endswith(".pdf"):
        print(f"Loading: {f}")
        text = load_pdf(f"/content/data/{f}")
        chunks = chunk_text(text)
        all_chunks.extend(chunks)

print(f"✅ Total chunks: {len(all_chunks)}")
print("\nSample chunk:")
print(all_chunks[0][:200])

Loading: Aishani Resume.pdf
✅ Total chunks: 2

Sample chunk:
CONTACT SOFT SKILLS LANGUAGES PROFILE 8918990682 Phone: aishanikundu25@gmail.com Email Address: TECH SKILLS Github: Teamwork Time Management Leadership Effective Communication Critical Thinking Englis


In [5]:
import chromadb
from chromadb.utils import embedding_functions

emb_fn = embedding_functions.DefaultEmbeddingFunction()

client = chromadb.Client()

try:
    client.delete_collection("rag_docs")
except:
    pass

collection = client.create_collection(
    name="rag_docs",
    embedding_function=emb_fn
)

batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    collection.add(
        documents=batch,
        ids=[f"chunk_{j}" for j in range(i, i+len(batch))]
    )

print(f"✅ Stored {collection.count()} chunks in ChromaDB!")

✅ Stored 2 chunks in ChromaDB!


In [6]:
def search(question, top_k=3):
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )
    print("=" * 50)
    print(f"Question: {question}")
    print("=" * 50)
    for i, doc in enumerate(results["documents"][0]):
        print(f"\nResult {i+1}:")
        print(doc[:300])
        print("-" * 30)

# Test
search("What are the skills mentioned?")

Question: What are the skills mentioned?

Result 1:
CONTACT SOFT SKILLS LANGUAGES PROFILE 8918990682 Phone: aishanikundu25@gmail.com Email Address: TECH SKILLS Github: Teamwork Time Management Leadership Effective Communication Critical Thinking English (Fluent) Bengali (Fluent) AISHANI KUNDU COMPUTER SCIENCE STUDENT EDUCATION Passionate about artifi
------------------------------

Result 2:
8.428 Class 12: 82% 2020-2021 2022-2023 Linkedin: www.linkedin.com/in/aishani-kundu Presented a research paper titled “A Survey on Paraphrase Detection in the Medical Field Using Deep Learning” at IEMECON 2025, where it was accepted for presentation. The paper is published. - Link
------------------------------


In [13]:
from groq import Groq

GROQ_API_KEY = "your_groq_api_key_here"  # paste your key

groq_client = Groq(api_key=GROQ_API_KEY)

def ask(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )
    context = "\n\n".join(results["documents"][0])

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # ✅ updated model!
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer questions based only on the given context. If the answer is not in the context, say 'I don't know'."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
        max_tokens=512,
        temperature=0.5
    )

    answer = response.choices[0].message.content
    print("=" * 50)
    print(f"Question: {question}")
    print("=" * 50)
    print(f"Answer: {answer}")
    print("=" * 50)
    return answer

# Test!
ask("What are the skills mentioned in this document?")

Question: What are the skills mentioned in this document?
Answer: The skills mentioned in this document are:

1. Time Management
2. Leadership
3. Effective Communication
4. Critical Thinking
5. Teamwork
6. Machine Learning
7. Data Analysis
8. Network Security
9. System Hardening
10. Product Development
11. Synthesizing AI/ML techniques with cybersecurity practices
12. Python (Intermediate)
13. Java (Intermediate)
14. C (Expert)
15. Data Structures
16. Algorithms
17. Deep Learning
18. NLP (Natural Language Processing)


'The skills mentioned in this document are:\n\n1. Time Management\n2. Leadership\n3. Effective Communication\n4. Critical Thinking\n5. Teamwork\n6. Machine Learning\n7. Data Analysis\n8. Network Security\n9. System Hardening\n10. Product Development\n11. Synthesizing AI/ML techniques with cybersecurity practices\n12. Python (Intermediate)\n13. Java (Intermediate)\n14. C (Expert)\n15. Data Structures\n16. Algorithms\n17. Deep Learning\n18. NLP (Natural Language Processing)'